# IS 6482 - Week 5 — Regression: Linear, Logistic, and Poisson

**Author:** Varun Gupta

**Date:** 8 March, 2026

This notebook develops a unified regression toolkit using both `statsmodels` and `scikit-learn`. By the end of this notebook, you should be able to:

- distinguish **inference** questions from **prediction** questions
- understand when to use **linear**, **logistic**, and **Poisson** regression
- interpret coefficients in each model family
- explain the practical difference between:
  - **`statsmodels`**: inference-oriented, rich summaries, hypothesis tests, standard errors, confidence intervals
  - **`scikit-learn`**: workflow-oriented, preprocessing, pipelines, and prediction APIs
- recognize two common `statsmodels` calling styles:
  - **formula API**: `y ~ x1 + x2`
  - **matrix API**: explicitly build `X` and `y`, closer to `scikit-learn`

## Imports and data

The notebook first looks for local CSV files in the current folder.  
If they are not present, it also checks `/mnt/data`.  
Only if local files are unavailable does it fall back to the public URLs.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.nonparametric.smoothers_lowess import lowess

from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_wine
from sklearn.linear_model import LinearRegression, LogisticRegression, PoissonRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from IPython.display import display

np.random.seed(42)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.figsize"] = (7, 5)
from scipy.stats import norm


In [ ]:
# URLs of datasets
advertising_url = 'https://www.statlearning.com/s/Advertising.csv'
credit_url = 'https://www.statlearning.com/s/Credit.csv'
default_url = 'https://raw.githubusercontent.com/intro-stat-learning/ISLP/main/ISLP/data/Default.csv'
bikeshare_url = 'https://raw.githubusercontent.com/intro-stat-learning/ISLP/main/ISLP/data/Bikeshare.csv'

# Read each data set.
advertising = pd.read_csv(advertising_url)
credit = pd.read_csv(credit_url)
default = pd.read_csv(default_url)
bikeshare = pd.read_csv(bikeshare_url)

# Drop the index column from Advertising if it is present.
if 'Unnamed: 0' in advertising.columns:
    advertising = advertising.drop(columns=['Unnamed: 0'])

advertising.head()

In [ ]:
# Always sanity-check the dimensions before modeling.
print("Advertising:", advertising.shape)
print("Credit:", credit.shape)
print("Default:", default.shape)
print("Bikeshare:", bikeshare.shape)

## 2. A Unifying perspective

A useful way to see how the four models we look at are part of one umbrella is through the **generalized linear model (GLM)** lens:

- **Linear regression**: Gaussian response + **identity** link
- **Binary logistic regression**: Bernoulli response + **logit** link
- **Multinomial logistic regression**: Multinomial response + **logit** link
- **Poisson regression**: Poisson response + **log** **link**

The modeling idea is the same each time:
1. The response distribution is the distribution of the target variable
2. The link function connects the mean of the response to a linear function of predictors

# Part I — Linear Regression

### A note on the two `statsmodels` styles used below
For most `statsmodels` examples, there are two methods of invoking the library:

1. **Formula API** (commented out): convenient and close to R syntax  
2. **Matrix API** (executed): explicitly build `X` and `y`, which makes the design matrix visible and feels closer to `scikit-learn`

Just to be consistent with sklearn, we will use the second

## 3. Simple Linear Regression (Advertising)

We begin with the model

$
\text{sales} = \beta_0 + \beta_1 \cdot \text{TV} + \varepsilon
$

This is the cleanest setting for interpreting slope, intercept, fitted values, and residuals.

### 3.1 OLS using `statsmodels`

In [ ]:
# Style 1 : Formula API: convenient and close to R.
# =============================================================
# ols_tv = smf.ols("sales ~ TV", data=advertising).fit()

In [ ]:
# Style 2 : Matrix API (executed): explicitly build y and X
# ==========================================================

y_sales = advertising["sales"]
# statsmodels.api.OLS does not include an intercept (constant term) by default.
# We must explicitly add it to our model's design matrix
X_tv_sm = sm.add_constant(advertising[["TV"]])

# Now call the fit function
# Note: in statsmodel, y goes first then X
#       in sklearn's .fit() X goes first then y
# Note: in statsmodel the data is passed to OLS() instance, and fit() does not take arguments
ols_tv = sm.OLS(y_sales, X_tv_sm).fit()

print(ols_tv.summary())

In [ ]:
# This function will produce a less verbose summary of the regression output
def coefficient_table_sm(result):
    ci = result.conf_int()
    if isinstance(ci, pd.DataFrame) and ci.shape[1] == 2:
        ci.columns = ["CI (lower)", "CI (upper)"]
    out = pd.DataFrame(
        {
            "Coefficient": result.params,
            "Std. error": result.bse,
            "t-statistic": result.tvalues,
            "p-value": result.pvalues,
        }
    )
    out = out.join(ci)
    return out.round(4)

In [ ]:
display(coefficient_table_sm(ols_tv))

In [ ]:
# Scatterplot of the raw data.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(advertising["TV"], advertising["sales"], alpha=0.75, s=35)

# Create a smooth x-grid so the fitted line looks clean.
x_grid = np.linspace(advertising["TV"].min(), advertising["TV"].max(), 200)
X_grid_sm = sm.add_constant(pd.DataFrame({"TV": x_grid}), has_constant="add")
y_grid = ols_tv.predict(X_grid_sm)
ax.plot(x_grid, y_grid, color="tab:orange", linewidth=2.5)

# Draw residual segments for a subset of points so students can see
# the vertical distance between observed sales and fitted sales.
sample_idx = np.linspace(0, len(advertising) - 1, 40, dtype=int)
for i in sample_idx:
    x_i = advertising.loc[i, "TV"]
    y_i = advertising.loc[i, "sales"]
    yhat_i = ols_tv.fittedvalues.iloc[i]
    ax.plot([x_i, x_i], [y_i, yhat_i], color="gray", alpha=0.5, linewidth=1)

ax.set_title("Sales vs TV advertising with fitted line and residuals")
ax.set_xlabel("TV advertising budget")
ax.set_ylabel("Sales")
plt.show()
plt.close('all')

In [ ]:
# Two important fit statistics
# - R^2: fraction of variation explained
# - Residual Standard Error: typical size of the residuals
simple_fit_stats = pd.Series({
    "R^2": ols_tv.rsquared,
    "Residual Standard Error": np.sqrt(ols_tv.scale)
} , name="value").round(4)

display(simple_fit_stats)

In [ ]:
def regression_overview(result):
  out = pd.DataFrame(
      {
          "value": [
              result.rsquared,
              np.sqrt(result.scale),
              result.rsquared_adj,
              result.fvalue,
              ]
          },
      index=["R-squared", "Residual Std. Error:", "Adj. R-squared", "F-statistic"],
      ).round(4)
  return out

In [ ]:
display(regression_overview(ols_tv))

In [ ]:
# Residual plot : Residuals vs. Predicted y
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(ols_tv.fittedvalues, ols_tv.resid, alpha=1, s=5)
ax.set_xlabel("Fitted values")
ax.set_ylabel("Residuals")
ax.set_title("Residuals vs. fitted values")
plt.show()
plt.close('all')

### `statsmodels` takeaway
`statsmodels` gives us the full inference table:
- coefficient estimates
- standard errors
- t-statistics
- p-values
- model fit statistics like \(R^2\) and residual standard error

### 3.2 OLS using `scikit-learn`

In [ ]:
# In scikit-learn, the pattern is always fit(X, y).
X_tv = advertising[["TV"]]
y_sales = advertising["sales"]

sklearn_tv = LinearRegression()
sklearn_tv.fit(X_tv, y_sales)

# scikit-learn exposes the fitted intercept and slope,
# and score(X, y) returns R^2 for linear regression.

print(f"intercept : {sklearn_tv.intercept_.round(4)}")
print(f"coef_[TV] : {sklearn_tv.coef_[0].round(4)}")
print(f"R^2 = score(X,y)  : {round(sklearn_tv.score(X_tv, y_sales), 4)}")

In [ ]:
# This function will return the coefficients from sklearn's fit in a nicer format
# by returning a Dataframe
def coefficient_table_sklearn(intercept, coef, feature_names):
    return (
        pd.DataFrame(
            {"term": ["Intercept", *feature_names], "coef": [intercept, *coef]}
        )
        .set_index("term")
        .round(4)
    )

In [ ]:
display(
    coefficient_table_sklearn(
        sklearn_tv.intercept_,
        sklearn_tv.coef_,
        X_tv.columns.to_list(),
    )
)

### `scikit-learn` takeaway
`scikit-learn` gives:
- `intercept_`
- `coef_`
- `score()` for \(R^2\)

But it does **not** give standard errors or p-values.  
That is the core philosophical difference:
- use **`statsmodels`** when you want **interpretation and inference**
- use **`scikit-learn`** when you want **prediction workflows and reusable pipelines**

We will focus on statsmodel for the rest of this notebook.

## 4. Multiple Regression (Advertising)

Now we fit

$
\text{sales} = \beta_0 + \beta_1 \cdot TV + \beta_2 \cdot radio + \beta_3 \cdot newspaper + \varepsilon
$

The key conceptual shift is that each coefficient is now interpreted **holding the other predictors fixed**.

In [ ]:
# Formula API (commented):
# ols_radio = smf.ols("sales ~ radio", data=advertising).fit()
# ols_news = smf.ols("sales ~ newspaper", data=advertising).fit()

# Matrix API:
X_radio_sm = sm.add_constant(advertising[["radio"]])
X_news_sm = sm.add_constant(advertising[["newspaper"]])

ols_radio = sm.OLS(y_sales, X_radio_sm).fit()
ols_news = sm.OLS(y_sales, X_news_sm).fit()

print("Simple regression: sales on radio")
display(coefficient_table_sm(ols_radio))
display(regression_overview(ols_radio))
print("="*80)

# Note that newspaper is a statistically significant predictor in this model
print("Simple regression: sales on newspaper")
display(coefficient_table_sm(ols_news))
display(regression_overview(ols_news))

In [ ]:
# Formula API (commented):
# ols_multi = smf.ols("sales ~ TV + radio + newspaper", data=advertising).fit()

# Matrix API:
X_multi_sm = sm.add_constant(advertising[["TV", "radio", "newspaper"]])
ols_multi = sm.OLS(y_sales, X_multi_sm).fit()

# In this model newspaper is not statistically significant
display(coefficient_table_sm(ols_multi))
display(regression_overview(ols_multi))

In [ ]:
# A correlation matrix helps explain why the multiple-regression story
# differs from the simple-regression story.
corr_media = advertising[["TV", "radio", "newspaper", "sales"]].corr().round(4)
sns.heatmap(corr_media, annot=True, cmap="coolwarm")
plt.show()
plt.close('all')

### Interpretation
In multiple regression, each coefficient is interpreted **holding the other predictors fixed**.

That is why the newspaper result changes:
- in a **simple** regression, newspaper looks helpful
- in the **multiple** regression, once TV and radio are included, newspaper is no longer significant

The correlation matrix helps explain this: newspaper is partly acting as a **surrogate** for other ad channels.

Another way of saying is that total advertising budget is **confounding variable** because it causes high newspaper budget and radio budget, and influences sales through radio budget but not newspaper budget. More on **graphical models** and **causal inference** later in this course

## 5. Interaction Terms (Advertising)

To allow synergy between media, we fit

$
\text{sales} = \beta_0 + \beta_1 TV + \beta_2 radio + \beta_3 (TV \times radio) + \varepsilon
$

An interaction lets the slope on one variable depend on the level of another variable.

In [ ]:
# Formula API (commented):
# ols_interaction = smf.ols("sales ~ TV * radio", data=advertising).fit()

# Matrix API:
X_interaction_sm = advertising[["TV", "radio"]].copy()
X_interaction_sm["TV:radio"] = advertising["TV"] * advertising["radio"]
X_interaction_sm = sm.add_constant(X_interaction_sm)

ols_interaction = sm.OLS(y_sales, X_interaction_sm).fit()

display(coefficient_table_sm(ols_interaction))
display(regression_overview(ols_interaction))

### Interaction interpretation
The coefficient on `TV:radio` tells us how the effect of one channel changes as the other channel changes.

So the slope on TV is no longer constant:
$
\frac{\partial \, E(\text{sales})}{\partial TV} = \beta_1 + \beta_3 \cdot radio
$

A positive interaction here means TV tends to work better when radio spending is also high. The $R^2$ of the model increased from 0.8972 to 0.9678 indicating the new model explains the data much better.

## 6. Qualitative Predictors (Credit)

This section introduces categorical predictors.

The key idea is that the formula interface in `statsmodels` can handle categories automatically, while `scikit-learn` typically requires explicit encoding.

In [ ]:
credit.head()

In [ ]:
# A scatter-matrix is a quick visual way to see pairwise relationships,
# marginal distributions, and possible collinearity.
# (this is like seaborn.pairplot() that we have used earlier)
credit_quant = credit[["Balance", "Age", "Cards", "Education", "Income", "Limit", "Rating"]]
axes = pd.plotting.scatter_matrix(
    credit_quant,
    figsize=(12, 12),
    diagonal='hist',
    alpha=0.6,
    s=20,
    grid=True
)

plt.suptitle("Pairwise relationships in the Credit data", y=0.92)
plt.show()
plt.close('all')

### Predictors with two levels
We will regress the response variable (Balance) on home ownership (Own) which has two levels (Yes, No)

In [ ]:
# Make the baseline category explicit for teaching:
# "No" will be the omitted category.
credit["Own"] = pd.Categorical(credit["Own"], categories=["No", "Yes"])
y_balance = credit["Balance"]

# Formula API (commented):
# ols_own = smf.ols("Balance ~ Own", data=credit).fit()

# Matrix API:
X_own_sm = pd.get_dummies(credit[["Own"]], drop_first=True, dtype=float)
X_own_sm = sm.add_constant(X_own_sm)
ols_own = sm.OLS(y_balance, X_own_sm).fit()

display(coefficient_table_sm(ols_own))
display(regression_overview(ols_own))

### Predictors with more than two levels
We will regress the response variable (Balance) on region (Region) which has three levels (East, West, South)

In [ ]:
# Make the region order explicit so East is the baseline.
credit["Region"] = pd.Categorical(credit["Region"], categories=["East", "South", "West"])

# Formula API (commented):
# ols_region = smf.ols("Balance ~ Region", data=credit).fit()

# Matrix API:
X_region_sm = pd.get_dummies(credit[["Region"]], drop_first=True, dtype=float)
X_region_sm = sm.add_constant(X_region_sm)
ols_region = sm.OLS(y_balance, X_region_sm).fit()

display(coefficient_table_sm(ols_region))
display(regression_overview(ols_region))

# Intepretation: average balance of a person from
# East = Coefficient(const)
# South = Coefficient(const) + Coefficient(Region_South)
# West = Coefficient(const) + Coefficient(Region_West)

### Baseline interpretation
With treatment coding, one category is the **baseline**.

For `Region`, we will use `East` as the baseline.  
That means:
- the intercept is the mean for the baseline group
- the dummy coefficients are differences relative to the baseline

### Handling categorical variables in `scikit-learn`

In [ ]:
# scikit-learn does not automatically turn a string column into dummies.
# We build that step explicitly with OneHotEncoder.
X_region = credit[["Region"]]
y_balance = credit["Balance"]

region_encoder = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", sparse_output=False), ["Region"])
    ],
    remainder="drop" # to drop all columns except Region
)

# fit_transform learns the encoding and returns the numeric design matrix.
X_region_encoded = region_encoder.fit_transform(X_region)
encoded_region_names = region_encoder.named_transformers_["cat"].get_feature_names_out(["Region"])

print("Encoded columns created by scikit-learn:")
display(pd.DataFrame(X_region_encoded, columns=encoded_region_names).head())

# After encoding, the regression step looks exactly like any other sklearn model.
sk_region = LinearRegression()
sk_region.fit(X_region_encoded, y_balance)

pd.Series(
    [sk_region.intercept_, *sk_region.coef_],
    index=["intercept_", *encoded_region_names]
).round(4)

### Categorical-variable takeaway
- **`statsmodels`** formula interface: `Balance ~ Region` is enough
- **`scikit-learn`**: we must explicitly create the dummy variables, usually with `OneHotEncoder(drop="first")`

For teaching, it is often useful to see both.

## 7. Interaction with a Qualitative Variable

Now we combine a quantitative variable (`Income`) with a qualitative variable (`Student`) and compare a model with and without an interaction.

In [ ]:
# Make the baseline explicit: Student = No will be the reference group.
credit["Student"] = pd.Categorical(credit["Student"], categories=["No", "Yes"])
student_yes = (credit["Student"] == "Yes").astype(int)

# Formula API (commented):
# ols_parallel = smf.ols("Balance ~ Income + Student", data=credit).fit()
# ols_student_inter = smf.ols("Balance ~ Income * Student", data=credit).fit()

# Matrix API without interaction: common slope, different intercept.
X_parallel_sm = pd.DataFrame({
    "Income": credit["Income"],
    "Student_Yes": student_yes
})
X_parallel_sm = sm.add_constant(X_parallel_sm)
ols_parallel = sm.OLS(y_balance, X_parallel_sm).fit()

# Matrix API with interaction: both slope and intercept may differ by student status.
X_student_inter_sm = pd.DataFrame({
    "Income": credit["Income"],
    "Student_Yes": student_yes,
    "Income:Student_Yes": credit["Income"] * student_yes
})
X_student_inter_sm = sm.add_constant(X_student_inter_sm)
ols_student_inter = sm.OLS(y_balance, X_student_inter_sm).fit()

display(coefficient_table_sm(ols_parallel))
display(coefficient_table_sm(ols_student_inter))

In [ ]:
# Plotting the two regression lines for Balance on Income, stratified by Student
income_grid = np.linspace(credit["Income"].min(), credit["Income"].max(), 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

# Left panel: no interaction -> parallel fitted lines.
sns.scatterplot(data=credit, x="Income", y="Balance", hue="Student", alpha=0.5, ax=axes[0], legend=False)
for student_value, label in zip([0, 1], ["No", "Yes"]):
    X_pred = pd.DataFrame({
        "const": 1.0,
        "Income": income_grid,
        "Student_Yes": student_value
    })
    axes[0].plot(income_grid, ols_parallel.predict(X_pred), linewidth=2, label=label)

axes[0].set_title("No interaction: separate intercepts, common slope")
axes[0].set_xlabel("Income")
axes[0].set_ylabel("Balance")

# Right panel: interaction -> lines can rotate as well as shift.
sns.scatterplot(data=credit, x="Income", y="Balance", hue="Student", alpha=0.5, ax=axes[1])
for student_value, label in zip([0, 1], ["No", "Yes"]):
    X_pred = pd.DataFrame({
        "const": 1.0,
        "Income": income_grid,
        "Student_Yes": student_value,
        "Income:Student_Yes": income_grid * student_value
    })
    axes[1].plot(income_grid, ols_student_inter.predict(X_pred), linewidth=2, label=label)

axes[1].set_title("Interaction: separate intercepts and slopes")
axes[1].set_xlabel("Income")
axes[1].set_ylabel("Balance")
plt.tight_layout()
plt.show()
plt.close('all')

With no interaction, the two fitted lines are parallel.  
With an interaction, both the intercept and the slope are allowed to differ by student status.

## 8. Collinearity Demonstration (Credit)

We focus on `Limit` and `Rating`, which are highly correlated in this data set.

The goal is to see that multicollinearity can make **individual coefficients unstable**, even when the model still predicts reasonably well.

In [ ]:
# Model 1: Limit enters with a relatively unrelated predictor (Age).
# Model 2: Limit enters with a highly correlated predictor (Rating).

# Formula API (commented):
# ols_col_1 = smf.ols("Balance ~ Age + Limit", data=credit).fit()
# ols_col_2 = smf.ols("Balance ~ Rating + Limit", data=credit).fit()

# Matrix API:
X_age_limit_sm = sm.add_constant(credit[["Age", "Limit"]])
X_rating_limit_sm = sm.add_constant(credit[["Rating", "Limit"]])

ols_col_1 = sm.OLS(y_balance, X_age_limit_sm).fit()
ols_col_2 = sm.OLS(y_balance, X_rating_limit_sm).fit()

print("Regression results for Balance ~ Age + Limit")
display(coefficient_table_sm(ols_col_1))
print('='*80)
print("Regression results for Balance ~ Rating + Limit")
display(coefficient_table_sm(ols_col_2))

In [ ]:
# Check the raw correlation among the main variables in the collinearity story.
credit[["Limit", "Rating", "Age"]].corr().round(4)

The following code illustrates that due to collinearity, the regression coefficients for Rating and Limit can be very noisy if both are included in the regression.

In [ ]:
# Refit the two models on many random subsamples.
# The point is not prediction accuracy here; it is coefficient stability.
coef_records_age_limit = []
coef_records_rating_limit = []

for seed in range(100):
    subset = credit.sample(frac=0.8, replace=False, random_state=seed)
    y_subset = subset["Balance"]

    X_age_limit_subset = sm.add_constant(subset[["Age", "Limit"]])
    X_rating_limit_subset = sm.add_constant(subset[["Rating", "Limit"]])

    fit_age_limit = sm.OLS(y_subset, X_age_limit_subset).fit()
    fit_rating_limit = sm.OLS(y_subset, X_rating_limit_subset).fit()

    coef_records_age_limit.append({
        "Limit": fit_age_limit.params["Limit"],
        "Age": fit_age_limit.params["Age"]
    })

    coef_records_rating_limit.append({
        "Limit": fit_rating_limit.params["Limit"],
        "Rating": fit_rating_limit.params["Rating"]
    })

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(
    x=[coef["Age"] for coef in coef_records_age_limit],
    y=[coef["Limit"] for coef in coef_records_age_limit],
    alpha=1, s=5)
axes[0].set_xlabel("Coefficient(Age)")
axes[0].set_ylabel("Coefficient(Limit)")
axes[0].set_title("Regression on Age and Limit")
axes[1].scatter(
    x=[coef["Rating"] for coef in coef_records_rating_limit],
    y=[coef["Limit"] for coef in coef_records_rating_limit],
    alpha=1, s=5)
axes[1].set_xlabel("Coefficient(Rating)")
axes[1].set_ylabel("Coefficient(Limit)")
axes[1].set_title("Regression on Rating and Limit")
plt.tight_layout()
plt.show()


### Intuition for multicollinearity
If two predictors move together, the model can explain the response using many nearby combinations of their coefficients.

That means:
- predictions can remain fairly stable
- **individual coefficients can become unstable**
- standard errors get larger
- p-values can become misleading

That is why `Limit` looks very strong with `Age`, but much less certain when `Rating` is included.

## 9. Regression Metrics

To connect regression to prediction, we now use a train/test split and compute several common metrics.

### Formulas
For test observations $(y_i, \hat y_i)$, let $\bar{ y}$ be the mean of the **test** response.
- Residual Sum of Squares : $RSS  =\sum (y_i - \hat y_i)^2 $
- R-squared:
$R^2 = 1 - \dfrac{\sum (y_i - \hat y_i)^2}{\sum (y_i - \bar y)^2}$
- Residual Standard Error:
$RSE = \sqrt{\dfrac{RSS}{n-p-1}}$  (usually reported on the training fit)
- Mean Absolute Error: $MAE = \dfrac{1}{n}\sum |y_i - \hat y_i|$
- Mean Absolute Percentage Error: $MAPE = \dfrac{100}{n}\sum \left|\dfrac{y_i - \hat y_i}{y_i}\right|$
- Relative Absolute Error: $RAE = \dfrac{\sum |y_i - \hat y_i|}{\sum |y_i - \bar y|}$
- Root Mean Squared Error: $RMSE = \sqrt{\dfrac{1}{n}\sum (y_i - \hat y_i)^2}$
- Root Mean Squared Percentation Error: $RMSPE = 100\sqrt{\dfrac{1}{n}\sum \left(\dfrac{y_i - \hat y_i}{y_i}\right)^2}$
- Root Relative Squared Error: $RRSE = \sqrt{\dfrac{\sum (y_i - \hat y_i)^2}{\sum (y_i - \bar y)^2}}$

In [ ]:
# Build a standard train/test split for prediction metrics.
X = advertising[["TV", "radio", "newspaper"]]
y = advertising["sales"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fit on the training set only.
reg = LinearRegression()
reg.fit(X_train, y_train)

# Evaluate on held-out data.
y_pred = reg.predict(X_test)

# Pieces that appear in several formulas below.
rss_test = np.sum((y_test - y_pred) ** 2)
tss_test = np.sum((y_test - y_test.mean()) ** 2)

# For RSE, we usually report the in-sample OLS quantity from the training fit.
X_train_sm = sm.add_constant(X_train)
ols_train = sm.OLS(y_train, X_train_sm).fit()

metrics_table = pd.Series({
    "R^2": r2_score(y_test, y_pred),
    "Residual Standard Error (training OLS)": np.sqrt(ols_train.scale),
    "MAE": mean_absolute_error(y_test, y_pred),
    "MAPE (%)": mean_absolute_percentage_error(y_test, y_pred) * 100,
    "RAE": np.sum(np.abs(y_test - y_pred)) / np.sum(np.abs(y_test - y_test.mean())),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
    "RMSPE (%)": np.sqrt(np.mean(((y_test - y_pred) / y_test) ** 2)) * 100,
    "RRSE": np.sqrt(rss_test / tss_test)
}).round(4)

metrics_table

`statsmodels` is especially useful when the goal is **explanation**; `scikit-learn` is especially useful when the goal is **out-of-sample prediction**.

# Part II — Logistic Regression (Binary)

## 10. Why Not Linear Regression for Classification?

We start with the `Default` data and compare a linear probability model to logistic regression.

In [ ]:
default.head()

In [ ]:
default.value_counts("default")

In [ ]:
# Visualization only -- you do not need to follow the code carefully

# Create a 0/1 response so both OLS and logistic regression can use it.
default["default01"] = (default["default"] == "Yes").astype(int)
default["income_k"] = default["income"] / 1000

# The classes are very imbalanced, so take a smaller sample of non-defaulters
# for a cleaner scatterplot
nondefault_sample = default[default["default"] == "No"].sample(n=1000, random_state=42)
default_subset = pd.concat([default[default["default"] == "Yes"], nondefault_sample])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].scatter(
    default_subset.loc[default_subset["default"] == "No", "balance"],
    default_subset.loc[default_subset["default"] == "No", "income"],
    alpha=0.4, s=18, label="No"
)
axes[0].scatter(
    default_subset.loc[default_subset["default"] == "Yes", "balance"],
    default_subset.loc[default_subset["default"] == "Yes", "income"],
    alpha=0.6, s=18, label="Yes"
)
axes[0].set_title("Balance vs income, colored by default status")
axes[0].set_xlabel("Balance")
axes[0].set_ylabel("Income")
axes[0].legend(title="Default")

sns.boxplot(data=default, x="default", y="balance", ax=axes[1])
axes[1].set_title("Balance by default status")
axes[1].set_xlabel("Default")
axes[1].set_ylabel("Balance")

sns.boxplot(data=default, x="default", y="income", ax=axes[2])
axes[2].set_title("Income by default status")
axes[2].set_xlabel("Default")
axes[2].set_ylabel("Income")

plt.tight_layout()
plt.show()
plt.close('all')

In [ ]:
# Linear probability model (OLS) versus logistic regression.
# Visualization only -- you do not need to follow the code carefully

# Formula API (commented):
# ols_prob = smf.ols("default01 ~ balance", data=default).fit()
# logit_prob = smf.logit("default01 ~ balance", data=default).fit(disp=0)

# Matrix API:
X_balance_sm = sm.add_constant(default[["balance"]])
y_default01 = default["default01"]

# Fitting an Linear Reggression Model
ols_prob = sm.OLS(y_default01, X_balance_sm).fit()
# Fitting a Logistic Regression Model
logit_prob = sm.Logit(y_default01, X_balance_sm).fit(disp=0)

# Prediction grid for a smooth curve.
balance_grid = np.linspace(default["balance"].min(), default["balance"].max(), 400)
pred_grid_sm = sm.add_constant(pd.DataFrame({"balance": balance_grid}), has_constant="add")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

# Left: OLS can go outside the [0, 1] range.
axes[0].scatter(default["balance"], default["default01"], alpha=0.08, s=10)
axes[0].plot(balance_grid, ols_prob.predict(pred_grid_sm), color="tab:orange", linewidth=2.5)
axes[0].set_title("Linear probability model")
axes[0].set_xlabel("Balance")
axes[0].set_ylabel("Estimated probability of default")

# Right: logistic regression bends into an S-shape and stays in [0, 1].
axes[1].scatter(default["balance"], default["default01"], alpha=0.08, s=10)
axes[1].plot(balance_grid, logit_prob.predict(pred_grid_sm), color="tab:green", linewidth=2.5)
axes[1].set_title("Logistic regression")
axes[1].set_xlabel("Balance")
axes[1].set_ylabel("Estimated probability of default")

plt.tight_layout()
plt.show()
plt.close('all')

### Main lesson
A linear probability model can predict values below 0 or above 1.  
Logistic regression keeps fitted probabilities inside **[0, 1]** by modeling the **log-odds**.

## 11. Logistic Regression on the Default Data

We now estimate a sequence of logistic regression models and interpret them using:
- coefficients on the log-odds scale
- odds ratios obtained by exponentiating coefficients
- comparisons between single-predictor and multiple-predictor fits

In [ ]:
# One-predictor logistic regression with balance.

# Formula API (commented):
# logit_balance = smf.logit("default01 ~ balance", data=default).fit(disp=0)

# Matrix API:
X_balance_sm = sm.add_constant(default[["balance"]])
logit_balance = sm.Logit(y_default01, X_balance_sm).fit(disp=0)

def coefficient_table_sm_logistic(result):
    out = pd.DataFrame({
    "Coefficient": result.params,
    "Std. error": result.bse,
    "z-statistic": result.tvalues,
    "p-value": result.pvalues
    })
    return out.round(4)

display(coefficient_table_sm_logistic(logit_balance))

In [ ]:
# Predicting probabilities for some rows of the training data with default = 1, and default = 0
display(logit_balance.predict(X_balance_sm[y_default01 == 1].iloc[0:5]))
display(logit_balance.predict(X_balance_sm[y_default01 == 0].iloc[0:5]))

In [ ]:
# One-predictor logistic regression with student status.
default["student"] = pd.Categorical(default["student"], categories=["No", "Yes"])
student_yes_default = (default["student"] == "Yes").astype(int)

# Formula API (commented):
# logit_student = smf.logit("default01 ~ student", data=default).fit(disp=0)

# Matrix API:
X_student_sm = sm.add_constant(pd.DataFrame({"student_Yes": student_yes_default}))
logit_student = sm.Logit(y_default01, X_student_sm).fit(disp=0)

display(coefficient_table_sm_logistic(logit_student))

In [ ]:
# Multiple logistic regression with balance, income, and student status.

# Formula API (commented):
# logit_multiple = smf.logit("default01 ~ balance + income_k + student", data=default).fit(disp=0)

# Matrix API:
X_default_sm = pd.DataFrame({
    "balance": default["balance"],
    "income_k": default["income_k"],
    "student_Yes": student_yes_default
})
X_default_sm = sm.add_constant(X_default_sm)
logit_multiple = sm.Logit(y_default01, X_default_sm).fit(disp=0)

display(coefficient_table_sm_logistic(logit_multiple))

### Interpreting logistic regression coefficients
For binary logistic regression,
$
\log\left(\frac{p(X)}{1-p(X)}\right) = \beta_0 + \beta_1 X_1 + \cdots + \beta_p X_p
$

So:
- coefficients live on the **log-odds** scale
- exponentiating a coefficient gives an **odds ratio**

### Why the sign flips for `student`
In the one-predictor model, students appear more likely to default.  
In the multiple-predictor model, the coefficient becomes negative.

Why? Because **students tend to carry higher balances**, and balance is strongly related to default.  
This is a classic example of **confounding**. When we add balance as a predictor, we "control" for it, and ask holding balance fixed, are students more likely to default.

### Scikit-Learn Logistic Regression code

In [ ]:
# scikit-learn follows the estimator API and regularizes by default.
# Here we turn regularization off to get something closer to statsmodels.
X_default = pd.get_dummies(default[["balance", "income_k", "student"]], drop_first=True)

# By default scikit-learn solves penalized logistic regression
# Here we are solving unpenalized by setting penalty=None
sk_logit = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=5000
)
sk_logit.fit(X_default, default["default01"])

sk_logit_table = pd.Series(
    [sk_logit.intercept_[0], *sk_logit.coef_[0]],
    index=["intercept_", *X_default.columns]
).round(4)

# This is just to confirm we get the same coefficients for the two
print("scikit-learn logistic coefficients")
display(sk_logit_table)

print("statsmodels logistic coefficients")
display(logit_multiple.params.round(4))

### Important `scikit-learn` note
`scikit-learn` logistic regression is **regularized by default**. To approximate the unregularized `statsmodels` fit, you can use:
- `penalty=None`, **or**
- a very large `C`

In this notebook we use `penalty=None`. If regularization is turned on, **feature scaling becomes important**. We will study regularization next week.

# Part III — Multiclass Logistic Regression

We now use the built-in **Wine** data from `scikit-learn`.

To keep the `statsmodels` demonstration compact and stable, we use three interpretable features:
- `alcohol`
- `color_intensity`
- `proline`

For multinomial logistic regression, the main conceptual idea is that each class is modeled relative to a baseline class.

In [ ]:
# Load the Wine data as a DataFrame so the columns keep readable names.
wine = load_wine(as_frame=True)
wine_df = wine.frame.copy()
wine_df.head()

In [ ]:
# Map target
wine_df["target_name"] = wine_df["target"].map(dict(enumerate(wine.target_names)))

# We will keep a compact, interpretable feature set for discussion
wine_features = ["alcohol", "color_intensity", "proline"]
X_wine = wine_df[wine_features]
y_wine = wine_df["target"]

display(X_wine.head())
display(y_wine.value_counts())


In [ ]:
# Multinomial logistic regression in scikit-learn.
# With penalty=None, this is an unregularized multinomial fit.
sk_multi = LogisticRegression(
    penalty=None,
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=5000
)
sk_multi.fit(X_wine, y_wine)

print("coef_.shape:", sk_multi.coef_.shape)
print("intercept_.shape:", sk_multi.intercept_.shape)

# Each row corresponds to one class; each column corresponds to one feature.
sk_multi_coef = pd.DataFrame(
    sk_multi.coef_,
    index=[f"class {c} ({wine.target_names[c]})" for c in sk_multi.classes_],
    columns=wine_features
).round(4)

display(sk_multi_coef)
display(pd.Series(sk_multi.intercept_, index=sk_multi_coef.index).round(4))

### Interpreting the `scikit-learn` multinomial coefficients
With 3 classes and 3 predictors:
- `coef_` has shape **(3, 3)**
- each row corresponds to a class
- each column corresponds to a predictor

The coefficients describe how predictors affect the relative log-probability of each class in the multinomial model.

In [ ]:
# Formula API (commented):
# mnlogit_results = smf.mnlogit("target ~ alcohol + color_intensity + proline", data=wine_df).fit(disp=0)

# Matrix API:
X_wine_sm = sm.add_constant(X_wine)
mnlogit_model = sm.MNLogit(y_wine, X_wine_sm)
mnlogit_results = mnlogit_model.fit(method="newton", maxiter=200, disp=False)

print(mnlogit_results.summary())

### Interpreting `statsmodels` MNLogit
`statsmodels` reports coefficients relative to a **baseline class**.

Conceptually, for class $k$,
$
\log \frac{P(Y=k)}{P(Y=\text{baseline})}
$
is modeled as a linear function of the predictors. Therefore in the above, the target=1 const coefficient is equal to intercept(class_1) - intercept(class_0) for the sklearn output.

# Part IV — Poisson Regression

## 12. Why Not OLS for Counts?

We use the Bikeshare data to see why count outcomes call for a different model family.

In [ ]:
# Make the category order explicit so the omitted baseline level is known.
month_order = ["Jan", "Feb", "March", "April", "May", "June", "July", "Aug", "Sept", "Oct", "Nov", "Dec"]
weather_order = ["clear", "cloudy/misty", "light rain/snow", "heavy rain/snow"]

bikeshare["mnth"] = pd.Categorical(bikeshare["mnth"], categories=month_order, ordered=True)
bikeshare["weathersit"] = pd.Categorical(bikeshare["weathersit"], categories=weather_order, ordered=True)
bikeshare["hr"] = pd.Categorical(bikeshare["hr"], categories=sorted(bikeshare["hr"].unique()), ordered=True)

# Formula API (commented):
# Option 1:
# ols_bike = smf.ols("bikers ~ C(mnth) + C(hr) + workingday + temp + C(weathersit)", data=bikeshare).fit()
# Option 2: to explicityly specify the refrence level for each categorical
# column ols_bike = smf.ols("bikers ~ C(mnth, Treatment(reference="Jan")) +
#     C(weathersit, Treatment(reference="clear")) + temp + workingday",
#     data=bikeshare).fit()

# Matrix API: explicitly create the dummy variables.
bike_X = pd.get_dummies(
    bikeshare[["mnth", "hr", "workingday", "temp", "weathersit"]],
    columns=["mnth", "hr", "weathersit"],
    drop_first=True,
    dtype=float
)
bike_X_sm = sm.add_constant(bike_X)

ols_bike = sm.OLS(bikeshare["bikers"], bike_X_sm).fit()

table_results = pd.DataFrame({
    "Coefficient": ols_bike.params,
    "Std. error": ols_bike.bse,
    "t-statistic": ols_bike.tvalues,
    "p-value": ols_bike.pvalues
}).round(4)

table_results.loc[
    ["const", "workingday", "temp",
     "weathersit_cloudy/misty",
     "weathersit_light rain/snow",
     "weathersit_heavy rain/snow"]
]

In [ ]:
# Look at the month and hour coefficients from the OLS count model.
month_coefs_ols = ols_bike.params[[c for c in ols_bike.params.index if c.startswith("mnth_")]]
hour_coefs_ols = ols_bike.params[[c for c in ols_bike.params.index if c.startswith("hr_")]]

month_labels = [idx.replace("mnth_", "") for idx in month_coefs_ols.index]
hour_labels = [idx.replace("hr_", "") for idx in hour_coefs_ols.index]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(month_labels, month_coefs_ols.values, marker="o")
axes[0].axhline(0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Month effects from OLS")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Coefficient")
axes[0].tick_params(axis="x", rotation=45)

axes[1].plot(hour_labels, hour_coefs_ols.values, marker="o")
axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Hour effects from OLS")
axes[1].set_xlabel("Hour")
axes[1].set_ylabel("Coefficient")

plt.tight_layout()
plt.show()
plt.close('all')

In [ ]:
# OLS can produce negative fitted values even though counts cannot be negative.
negative_share = (ols_bike.fittedvalues < 0).mean()
print(f"Share of negative fitted values from OLS: {negative_share:.3%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)

jitter = np.random.uniform(-0.15, 0.15, size=len(bikeshare))

axes[0].scatter(bikeshare["hr"].astype(int) + jitter, bikeshare["bikers"], alpha=0.08, s=10)
smooth_left = lowess(bikeshare["bikers"], bikeshare["hr"].astype(int), frac=0.15, return_sorted=True)
axes[0].plot(smooth_left[:, 0], smooth_left[:, 1], color="tab:green", linewidth=2.5)
axes[0].set_title("Counts by hour")
axes[0].set_xlabel("Hour")
axes[0].set_ylabel("Number of bikers")

# After a log transform, the variance of number of bikers (on log scale) shows
# somewhat constant variance across hours of day
axes[1].scatter(bikeshare["hr"].astype(int) + jitter, np.log(bikeshare["bikers"]), alpha=0.08, s=10)
smooth_right = lowess(np.log(bikeshare["bikers"]), bikeshare["hr"].astype(int), frac=0.15, return_sorted=True)
axes[1].plot(smooth_right[:, 0], smooth_right[:, 1], color="tab:green", linewidth=2.5)
axes[1].set_title("Log-counts by hour")
axes[1].set_xlabel("Hour")
axes[1].set_ylabel("log(Number of bikers)")

plt.tight_layout()
plt.show()
plt.close('all')

### Why OLS is awkward for counts
Count responses create three problems for ordinary least squares:

1. **negative fitted values** are possible  
2. the response is **integer-valued**, not continuous  
3. the variance often grows with the mean, so constant-variance errors are unrealistic

A log transformation helps, but it changes the interpretation and is not always ideal.

## 13. Poisson Regression (Bikeshare)

Now we fit a Poisson GLM:

$
\log(E[Y \mid X]) = \beta_0 + \beta_1 X_1 + \cdots + \beta_p X_p
$

The log link ensures that fitted mean counts stay positive.

In [ ]:
# Formula API (commented):
# poisson_bike = smf.glm(
#     "bikers ~ C(mnth) + C(hr) + workingday + temp + C(weathersit)",
#     data=bikeshare,
#     family=sm.families.Poisson()
# ).fit()

# Matrix API:
poisson_bike = sm.GLM(
    bikeshare["bikers"],
    bike_X_sm,
    family=sm.families.Poisson()
).fit()

table_results_Poisson = pd.DataFrame({
    "Coefficient": poisson_bike.params,
    "Std. error": poisson_bike.bse,
    "z-statistic": poisson_bike.tvalues,
    "p-value": poisson_bike.pvalues,
    "exp(coef)": np.exp(poisson_bike.params)
}).round(4)

table_results_Poisson_subset = table_results_Poisson.loc[
    ["const", "workingday", "temp",
     "weathersit_cloudy/misty",
     "weathersit_light rain/snow",
     "weathersit_heavy rain/snow"]
]
table_results_Poisson_subset

In [ ]:
# Compare the estimated month and hour effects under a Poisson GLM.
# The number we are seeing are increase in log-odds compared to baseline
#     (January for month, 0hrs for hour of day)
month_coefs_pois = poisson_bike.params[[c for c in poisson_bike.params.index if c.startswith("mnth_")]]
hour_coefs_pois = poisson_bike.params[[c for c in poisson_bike.params.index if c.startswith("hr_")]]

month_labels = [idx.replace("mnth_", "") for idx in month_coefs_pois.index]
hour_labels = [idx.replace("hr_", "") for idx in hour_coefs_pois.index]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(month_labels, month_coefs_pois.values, marker="o")
axes[0].axhline(0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Month effects from Poisson regression")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Coefficient")
axes[0].tick_params(axis="x", rotation=45)

axes[1].plot(hour_labels, hour_coefs_pois.values, marker="o")
axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Hour effects from Poisson regression")
axes[1].set_xlabel("Hour")
axes[1].set_ylabel("Coefficient")

plt.tight_layout()
plt.show()
plt.close('all')

### Poisson interpretation
Because the link is log,
$
E[Y \mid X] = \exp(X\beta)
$

So a one-unit increase in predictor $X_j$ changes the mean count **multiplicatively** by:
$\exp(\beta_j)$

Example:
- if $\beta_j = -0.08$, then the mean count is multiplied by $e^{-0.08} \approx 0.923$
- that means about a **7.7% decrease**

### Optional : Poisson regression using `scikit-learn`

In [ ]:
# Optional - Poisson Regression using scikit-learn

bike_num_X = bikeshare[["workingday", "temp"]]
bike_num_y = bikeshare["bikers"]

# A compact scikit-learn example with only numeric predictors.
# The purpose here is to show the estimator API, not to build the richest count model.
sk_pois = PoissonRegressor(alpha=0, solver="newton-cholesky", max_iter=300)
sk_pois.fit(bike_num_X, bike_num_y)

# Fit the same numeric-only specification in statsmodels for a clean comparison.
sm_pois_subset = sm.GLM(
    bike_num_y,
    sm.add_constant(bike_num_X),
    family=sm.families.Poisson()
).fit()

print("scikit-learn PoissonRegressor coef_")
display(pd.Series(sk_pois.coef_, index=bike_num_X.columns).round(4))

print("scikit-learn intercept_")
display(pd.Series({"intercept_": sk_pois.intercept_}).round(4))

poisson_compare = pd.DataFrame({
    "statsmodels": [sm_pois_subset.params["const"], sm_pois_subset.params["workingday"], sm_pois_subset.params["temp"]],
    "scikit-learn": [sk_pois.intercept_, sk_pois.coef_[0], sk_pois.coef_[1]],
}, index=["Intercept", "workingday", "temp"]).round(4)

print("Same numeric-only specification in both libraries")
display(poisson_compare)

### Important `scikit-learn` note
`PoissonRegressor` uses **L2 regularization by default** through `alpha=1`.

To approximate an unregularized fit comparable to `statsmodels`, set:
- `alpha=0`

In this notebook, `statsmodels` is used for the richer inference-oriented count model, while `scikit-learn` is used to show the estimator-style workflow.

### Brief note on overdispersion
A Poisson model assumes:
$
E(Y \mid X) = Variance(Y \mid X)
$

That is convenient, but real count data often have **more variance than the mean**. That phenomenon is called **overdispersion**.

We will only flag it here; we will not study remedies in this notebook.

## Final Summary

A useful way to see how the four models we look at are part of one umbrella is through the **generalized linear model (GLM)** lens:

- **Linear regression**: Gaussian response + **identity** link
- **Binary logistic regression**: Bernoulli response + **logit** link
- **Multinomial logistic regression**: Multinomial response + **logit** link
- **Poisson regression**: Poisson response + **log** **link**

The modeling idea is the same each time:
1. The response distribution is the distribution of the target variable
2. The link function connects the mean of the response to a linear function of predictors

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Linear regression",
        "Binary logistic regression",
        "Multinomial logistic regression",
        "Poisson regression"
    ],
    "Distribution": [
        "Gaussian",
        "Bernoulli",
        "Multinomial",
        "Poisson"
    ],
    "Link": [
        "Identity",
        "Logit",
        "Logit",
        "Log"
    ],
    "statsmodels gives": [
        "coef, SE, t, p, CI, R^2, RSE",
        "coef, SE, z, p, odds-ratio interpretation",
        "coef, SE, z, p, baseline-relative inference",
        "coef, SE, z, p, exp(coef) interpretation"
    ],
    "sklearn gives": [
        "fit/predict workflow, coef_, intercept_, R^2",
        "fit/predict workflow, coef_, intercept_, regularization by default",
        "multinomial workflow, coef_, intercept_",
        "pipeline workflow, coef_, intercept_, regularization by default"
    ]
})

comparison

## Coming up next week

- multicollinearity diagnostics
- residual analysis
- heteroskedasticity
- regularization (**Ridge/Lasso**)
- feature selection (forward, backward)
